# OCR Benchmark

**Estimated times:**
- Docling (all pages): ~1.5 hours
- Marker (no LLM): ~8 hours
- Marker + LLM: ~10 hours

## 1. Setup Environment

In [ ]:
# Clone the repository
!git clone https://github.com/buinguyenkhai/stock-report-agent-20251.git
%cd stock-report-agent-20251

In [ ]:
# Install dependencies
!pip install -q pymupdf datasets pillow jiwer tenacity python-dotenv pydantic-settings
!pip install -q docling docling-core
!pip install -q marker-pdf surya-ocr

In [ ]:
# Set environment variables
import os

os.environ['OPENROUTER_API_KEY'] = 'YOUR_OPENROUTER_API_KEY' 

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU count: {torch.cuda.device_count()}")

## 2. Quick Validation (5 minutes)

In [ ]:
# Test imports work
import sys
sys.path.insert(0, '.')

from evaluation import PageLevelBenchmark
print("Imports OK")

In [ ]:
# Quick validation: 1 page each
!python -m evaluation.ocr_benchmark.page_level_benchmark --engine docling --companies AAA --max-pages 1 --output results/validation_docling.json

In [ ]:
# Check validation results
import json
with open('results/validation_docling.json') as f:
    result = json.load(f)
    print(f"Docling NumF1: {result['overall_avg_number_f1']:.2%}")
    print("Expected: > 80%")

## 3. Full Docling Benchmark (~1.5 hours)

In [ ]:
# Run Docling on all companies, all pages
!python -m evaluation.ocr_benchmark.page_level_benchmark \
    --engine docling \
    --output results/docling_full.json

In [ ]:
# View Docling results
with open('results/docling_full.json') as f:
    result = json.load(f)
    print("DOCLING FULL RESULTS")
    print(f"Companies: {result['total_companies']}")
    print(f"Pages: {result['total_pages']}")
    print(f"NumF1: {result['overall_avg_number_f1']:.2%} ± {result['overall_std_number_f1']:.2%}")
    print(f"FA-CER: {result['overall_avg_format_agnostic_cer']:.2%}")
    print(f"Word Recall: {result['overall_avg_content_word_recall']:.2%}")
    print(f"Time: {result['total_time_seconds']/60:.1f} min")

## 4. Full Marker Benchmark (~8 hours)

In [ ]:
# Run Marker (no LLM) on all companies, all pages
!python -m evaluation.ocr_benchmark.page_level_benchmark \
    --engine marker \
    --output results/marker_full.json

In [ ]:
# View Marker results
with open('results/marker_full.json') as f:
    result = json.load(f)
    print("MARKER (NO LLM) FULL RESULTS")
    print(f"Companies: {result['total_companies']}")
    print(f"Pages: {result['total_pages']}")
    print(f"NumF1: {result['overall_avg_number_f1']:.2%} ± {result['overall_std_number_f1']:.2%}")
    print(f"FA-CER: {result['overall_avg_format_agnostic_cer']:.2%}")
    print(f"Word Recall: {result['overall_avg_content_word_recall']:.2%}")
    print(f"Time: {result['total_time_seconds']/3600:.1f} hours")

## 5. Marker + LLM Benchmark (~10 hours)

**⚠️ Requires OPENROUTER_API_KEY. Run overnight!**

In [ ]:
# Run Marker with LLM on all companies, all pages
!python -m evaluation.ocr_benchmark.page_level_benchmark \
    --engine marker \
    --marker-llm \
    --output results/marker_llm_full.json

In [ ]:
# View Marker + LLM results
with open('results/marker_llm_full.json') as f:
    result = json.load(f)
    print("MARKER + LLM FULL RESULTS")
    print(f"Companies: {result['total_companies']}")
    print(f"Pages: {result['total_pages']}")
    print(f"NumF1: {result['overall_avg_number_f1']:.2%} ± {result['overall_std_number_f1']:.2%}")
    print(f"FA-CER: {result['overall_avg_format_agnostic_cer']:.2%}")
    print(f"Word Recall: {result['overall_avg_content_word_recall']:.2%}")
    print(f"Time: {result['total_time_seconds']/3600:.1f} hours")

## 6. Summary Comparison

In [ ]:
import json
import pandas as pd

results = []
files = [
    ('Docling', 'results/docling_full.json'),
    ('Marker', 'results/marker_full.json'),
    ('Marker + LLM', 'results/marker_llm_full.json'),
]

for name, path in files:
    try:
        with open(path) as f:
            r = json.load(f)
            results.append({
                'Engine': name,
                'Pages': r['total_pages'],
                'NumF1': f"{r['overall_avg_number_f1']:.2%}",
                'FA-CER': f"{r['overall_avg_format_agnostic_cer']:.2%}",
                'Word Recall': f"{r['overall_avg_content_word_recall']:.2%}",
                'Time (min)': f"{r['total_time_seconds']/60:.1f}",
            })
    except FileNotFoundError:
        print(f"⚠️ {path} not found")

df = pd.DataFrame(results)
print("OCR BENCHMARK COMPARISON")
print(df.to_markdown(index=False))

## 7. Download Results

In [ ]:
# Zip results for download
!zip -r benchmark_results.zip results/

# In Kaggle, use the "Output" tab to download benchmark_results.zip